# Project: Walk-forward validation ⏩
Random K-fold cross-validation on time series **lies**: it trains on tomorrow to predict today. A model that knows the *date* (a time index and the season) can simply interpolate between the neighbouring days it saw in training, which is impossible when you really forecast. Measure the lie on Aalborg temperature data: random K-fold vs. walk-forward splits vs. a true future holdout (2024).

Topic: [[Time Series Validation and Features]], [[Cross-Validation and Model Selection]] · guide note: [[Project - Walk-Forward Validation]] · data as in [[Project - Forecasting Aalborg Temperature]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import json, pathlib, urllib.parse, urllib.request, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import KFold
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
f = pathlib.Path("data/dmi_06030_temp_2018_2024.csv"); f.parent.mkdir(exist_ok=True)
if not f.exists():
    q = dict(stationId="06030", parameterId="temp_mean_past1h", datetime="2018-01-01T00:00:00Z/2025-01-01T00:00:00Z", limit=300000)
    d = json.load(urllib.request.urlopen("https://opendataapi.dmi.dk/v2/metObs/collections/observation/items?" + urllib.parse.urlencode(q)))
    pd.DataFrame([x["properties"] for x in d["features"]])[["observed", "value"]].to_csv(f, index=False)
t = pd.read_csv(f); s = t.assign(d=pd.to_datetime(t.observed)).set_index("d").value.sort_index().resample("D").mean().interpolate(limit=3).dropna()
df = pd.DataFrame({"y": s})                                                  # target: the day's mean temperature
df["t"] = np.arange(len(df))                                                 # time index (days since 2018-01-01)
df["doy_sin"], df["doy_cos"] = np.sin(2 * np.pi * s.index.dayofyear / 365.25), np.cos(2 * np.pi * s.index.dayofyear / 365.25)
dev, hold = df[:"2023-12-31"], df["2024-01-01":]
X, y = dev.drop(columns="y").to_numpy(), dev.y.to_numpy()
print(f"{len(dev)} development days (2018–2023), {len(hold)} holdout days (2024); features: time index + season")

## 1. Walk-forward splits
Return a list of `(train_idx, test_idx)`: the last `n_splits × test_size` rows are cut into consecutive test blocks; each block's training set is **everything before it**, minus a `gap` of rows just before the test block (to avoid overlap through lag features).

In [ ]:
def walk_forward_splits(n, n_splits=5, test_size=180, gap=7):
    # TODO 1: expanding-window splits, test blocks at the end, gap before each test block
    raise NotImplementedError  # TODO 1

## 2. Cross-validated MAE
For each split, fit a **fresh clone** of the model on the training rows, compute the MAE on the test rows; return the mean.

In [ ]:
def cv_mae(model, X, y, splits):
    # TODO 2
    raise NotImplementedError  # TODO 2

In [ ]:
def _t_wf():
    sp = walk_forward_splits(100, n_splits=3, test_size=10, gap=2)
    return [(tr.max(), te.min(), te.max()) for tr, te in sp] == [(67, 70, 79), (77, 80, 89), (87, 90, 99)] and all(tr.min() == 0 for tr, _ in sp)
check("walk-forward splits", _t_wf, "n=100, 3×10 test rows, gap 2: tests 70–79, 80–89, 90–99; training ends 2 rows before each test block.")
if ready("walk-forward splits"):
    try:
        model = ExtraTreesRegressor(n_estimators=200, min_samples_leaf=3, random_state=0, n_jobs=-1)
        random_cv = cv_mae(model, X, y, list(KFold(5, shuffle=True, random_state=0).split(X)))
        wf_cv = cv_mae(model, X, y, walk_forward_splits(len(X)))
        m = clone(model).fit(X, y); true_err = float(np.mean(np.abs(m.predict(hold.drop(columns="y").to_numpy()) - hold.y.to_numpy())))
        print(f"random 5-fold CV MAE   {random_cv:.3f} °C\nwalk-forward CV MAE    {wf_cv:.3f} °C\nTRUE 2024 holdout MAE  {true_err:.3f} °C")
        pd.Series({"random K-fold": random_cv, "walk-forward": wf_cv, "true future (2024)": true_err}).plot.bar(rot=0, figsize=(6, 3), title="estimated vs. real error"); plt.show()
        check("random CV is optimistic", lambda: random_cv < wf_cv and abs(wf_cv - true_err) < abs(random_cv - true_err),
              "Random K-fold should underestimate the error, and walk-forward should be closer to the true future error.")
    except NotImplementedError:
        print("❌ random CV is optimistic: not implemented yet – fill in the TODO above.")

<details><summary>▶ Solution</summary>

```python
def walk_forward_splits(n, n_splits=5, test_size=180, gap=7):
    splits = []
    for k in range(n_splits):
        test_end = n - (n_splits - 1 - k) * test_size
        test_start = test_end - test_size
        train_end = test_start - gap
        splits.append((np.arange(0, train_end), np.arange(test_start, test_end)))
    return splits
```

```python
def cv_mae(model, X, y, splits):
    errs = []
    for tr, te in splits:
        m = clone(model).fit(X[tr], y[tr])
        errs.append(np.mean(np.abs(m.predict(X[te]) - y[te])))
    return float(np.mean(errs))
```
</details>

## Stretch goals
- Add yesterday's temperature as a feature (a real forecasting setup): how much do the three estimates move, and does random CV still lie as much?
- Compare `sklearn.model_selection.TimeSeriesSplit` with your splits.
- Use walk-forward CV to tune `min_samples_leaf`. Does the choice differ from random CV's?